# Pré-processamento — Dataset CSEDM / ProgSnap2 v6

Transforma os dados brutos do CSEDM em sequências KT prontas para BKT, DKT e Code-DKT.  
Metodologia: fase de *Data Preparation* do EDM Process (Kalita et al., 2025).

---

## Splits do Dataset

O CSEDM disponibiliza dois semestres independentes, cada um com Train/Test próprios:

| Split | Semestre | Estudantes | Uso recomendado |
|-------|----------|------------|-----------------|
| `All/` | Fall-2019 (set–dez) | 506 | EDA exploratória completa |
| `Release/` | Spring-2019 (fev–mai) | 329 | **Comparação reproduzível com Shi et al. (2022)** |

As populações **não se sobrepõem**: SubjectIDs de `All/` ≠ SubjectIDs de `Release/`.  
Para replicar os resultados do paper Code-DKT (AUC ~74% em A1), usar sempre `Release/Train` para treino e `Release/Test` para teste.  
O benchmark de reprodutibilidade é **23.70% de tentativas corretas em Release/Train** (Shi et al. (2022) reporta 23.68% — margem de arredondamento esperada).

**Referência:** Shi et al. (2022); Pankiewicz, Shi & Baker (2025).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

NOTEBOOK_DIR = Path().resolve()
ROOT = NOTEBOOK_DIR.parent
DATA_ROOT = ROOT / 'data' / 'CSEDM'
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

# Adicionar src/ ao path para importar data_loader
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_loader import load_main_table, load_labels

print(f'SEED = {SEED}')
print(f'DATA_ROOT = {DATA_ROOT}')
print(f'DATA_ROOT existe: {DATA_ROOT.exists()}')

SEED = 42
DATA_ROOT = /home/leokuntz/Documents/repositories/studies/tcc.edm.kt/data/CSEDM
DATA_ROOT existe: True


---

## 1 — Setup e Carregamento dos Dados

### 1.1 — Carregamento dos splits

**Contexto:** O primeiro passo de qualquer pipeline de pré-processamento é carregar os dados e confirmar que os splits têm as dimensões esperadas. Aqui carregamos os quatro splits relevantes: `All/Data` (EDA completa), `Release/Train` e `Release/Test` (treinamento e avaliação reproduzíveis conforme o protocolo de Shi et al. (2022)).  
**Hipótese:** `All/Data` deve conter ~360 mil eventos e 506 estudantes. `Release/Train` deve ter ~134 mil eventos com 23.70% de corretos em `Run.Program`. `Release/Test` deve ter ~32 mil eventos e 83 estudantes.  
**Referência:** Price et al. (2020); Shi et al. (2022).

In [2]:
# Carregar todos os splits
all_main        = load_main_table('all',           DATA_ROOT)
release_train   = load_main_table('release_train', DATA_ROOT)
release_test    = load_main_table('release_test',  DATA_ROOT)

# Labels (predição final do pipeline KT)
early_train = load_labels('release_train', DATA_ROOT, which='early')
late_train  = load_labels('release_train', DATA_ROOT, which='late')
early_test  = load_labels('release_test',  DATA_ROOT, which='early')
late_test   = load_labels('release_test',  DATA_ROOT, which='late')

splits = {
    'All/Data':        all_main,
    'Release/Train':   release_train,
    'Release/Test':    release_test,
}

print(f'{"Split":<20} {"Eventos":>10} {"Estudantes":>12} {"Assignments":>13}')
print('-' * 58)
for name, df in splits.items():
    n_events    = len(df)
    n_subjects  = df['SubjectID'].nunique()
    n_assign    = df['AssignmentID'].nunique()
    print(f'{name:<20} {n_events:>10,} {n_subjects:>12,} {n_assign:>13,}')

Split                   Eventos   Estudantes   Assignments
----------------------------------------------------------
All/Data                360,176          506             5
Release/Train           134,508          246             5
Release/Test             32,372           83             3


**Achado:** Os splits têm as dimensões esperadas — `All/Data` com ~360k eventos e 506 estudantes; `Release/Train` com ~134k e 246 estudantes; `Release/Test` com ~32k e 83 estudantes. Populações sem sobreposição (semestres distintos).  
**Implicação para modelagem:** Toda comparação com os resultados de Shi et al. (2022) deve usar exclusivamente `Release/Train` e `Release/Test`. O split `All/` é reservado para análises exploratórias e não deve ser misturado com `Release/`.

### 1.2 — Benchmark de reprodutibilidade

**Contexto:** Shi et al. (2022) reporta que o dataset de treinamento contém 23.68% de tentativas corretas (Run.Program com Score == 1.0). Confirmar este valor é o primeiro teste de reprodutibilidade do pipeline.  
**Hipótese:** A taxa calculada em `Release/Train` deve ser ~23.68–23.70%, refletindo apenas diferença de arredondamento entre os papers.  
**Referência:** Shi et al. (2022) — Section 4.1, "23.68% of the attempts are correct".

In [3]:
runs_train = release_train[release_train['EventType'] == 'Run.Program'].copy()
runs_test  = release_test[release_test['EventType']  == 'Run.Program'].copy()

correct_rate_train = (runs_train['Score'] == 1.0).mean()
correct_rate_test  = (runs_test['Score']  == 1.0).mean()

print(f'Release/Train — Run.Program: {len(runs_train):,} eventos')
print(f'  Taxa de corretos (Score == 1.0): {correct_rate_train:.4%}')
print(f'  Paper Shi et al. (2022):          23.6800%')
print(f'  Divergência:                      {abs(correct_rate_train - 0.2368):.4%}')
print()
print(f'Release/Test — Run.Program: {len(runs_test):,} eventos')
print(f'  Taxa de corretos (Score == 1.0): {correct_rate_test:.4%}')

Release/Train — Run.Program: 46,825 eventos
  Taxa de corretos (Score == 1.0): 23.7010%
  Paper Shi et al. (2022):          23.6800%
  Divergência:                      0.0210%

Release/Test — Run.Program: 10,845 eventos
  Taxa de corretos (Score == 1.0): 21.1803%


**Achado:** A taxa de corretos em `Release/Train` é ~23.70%, a menos de 0.02pp do valor de 23.68% reportado por Shi et al. (2022) — diferença de arredondamento esperada. O benchmark de reprodutibilidade é confirmado.  
**Implicação para modelagem:** O dataset está íntegro para reprodução do paper. A baixa proporção de acertos (~23.7%) justifica o uso de AUC como métrica primária em vez de acurácia — uma predição trivial "sempre errado" teria acurácia de 76.3%, mas AUC de ~50%.

### 1.3 — Estrutura e tipos de dados

**Contexto:** Verificar se as colunas críticas estão presentes e com tipos corretos. Os modelos KT dependem de `SubjectID`, `AssignmentID`, `ProblemID`, `ServerTimestamp` (ordem cronológica), `EventType` e `Score`.  
**Hipótese:** Todas as colunas críticas devem estar presentes em `Release/Train`. O `Release/` tem uma coluna extra `CourseSectionID` e `SourceLocation` ausentes em `All/`.  
**Referência:** Price et al. (2020) — ProgSnap2 v6 specification.

In [4]:
CRITICAL_COLS = ['SubjectID', 'AssignmentID', 'ProblemID', 'ServerTimestamp',
                 'EventType', 'Score', 'CodeStateID']

print('=== Release/Train — dtypes e nulos ===')
info = pd.DataFrame({
    'dtype':  release_train.dtypes.astype(str),
    'nulls':  release_train.isna().sum(),
    'null_%': (release_train.isna().mean() * 100).round(2),
})
print(info.to_string())
print()

missing = [c for c in CRITICAL_COLS if c not in release_train.columns]
if missing:
    print(f'AVISO — colunas críticas ausentes: {missing}')
else:
    print('OK — todas as colunas críticas presentes em Release/Train')

# Score só existe em Run.Program — nulo em Compile e Compile.Error é esperado
score_null_outside_run = release_train[
    (release_train['EventType'] != 'Run.Program') & release_train['Score'].notna()
]
print(f'Score não-nulo fora de Run.Program: {len(score_null_outside_run)} (esperado: 0)')

=== Release/Train — dtypes e nulos ===
                                         dtype  nulls  null_%
Order                                    int64      0    0.00
SubjectID                                  str      0    0.00
ToolInstances                              str      0    0.00
ServerTimestamp            datetime64[us, UTC]      0    0.00
ServerTimezone                             str      0    0.00
CourseID                                   str      0    0.00
CourseSectionID                          int64      0    0.00
AssignmentID                             Int64      0    0.00
ProblemID                                Int64      0    0.00
CodeStateID                                str      0    0.00
IsEventOrderingConsistent                 bool      0    0.00
EventType                                  str      0    0.00
Score                                  float64  87683   65.19
Compile.Result                             str  87683   65.19
CompileMessageType             

**Achado:** Todas as colunas críticas estão presentes. `Score` é nulo em `Compile` e `Compile.Error` (comportamento esperado — Score só existe em `Run.Program`). `ServerTimestamp` foi convertido para `datetime64[ns, UTC]` pelo `load_main_table`, garantindo ordenação cronológica correta.  
**Implicação para modelagem:** Nenhuma engenharia de colunas adicional é necessária para o carregamento. O `load_main_table` de `src/data_loader.py` entrega o DataFrame pronto para as etapas de filtragem (tarefa 2).

---

## 2 — Filtragem por Modelo

### 2.1 — filter_for_bkt_dkt e filter_for_code_dkt

**Contexto:** BKT e DKT processam apenas sequências de acerto/erro sem informação do código-fonte; portanto, entram na sequência somente eventos `Run.Program` (execuções com Score). Code-DKT, por outro lado, extrai embeddings AST a cada submissão — incluindo código com erros de compilação — e por isso adiciona os eventos `Compile.Error` como `correct=0`. Essa distinção de protocolo é central para reproduzir os resultados de Shi et al. (2022) e Pankiewicz, Shi & Baker (2025).  
**Hipótese:** Após a filtragem, BKT/DKT devem conter apenas `Run.Program` (~46k eventos em Release/Train); Code-DKT deve conter `Run.Program` + `Compile.Error` (~87k eventos). A proporção de corretos em Code-DKT deve ser menor que em BKT/DKT, pois os `Compile.Error` contribuem exclusivamente com `correct=0`.  
**Referência:** Shi et al. (2022) — Section 3.1, "correct=1 when all tests pass"; Pankiewicz, Shi & Baker (2025) — srcML-DKT, EDM 2025.

In [5]:
from data_loader import filter_for_bkt_dkt, filter_for_code_dkt

# --- Filtragem ---
bkt_dkt_train = filter_for_bkt_dkt(release_train)
bkt_dkt_test  = filter_for_bkt_dkt(release_test)

code_dkt_train = filter_for_code_dkt(release_train)
code_dkt_test  = filter_for_code_dkt(release_test)

# --- Verificação de EventType: nenhum outro tipo deve ter passado ---
assert set(bkt_dkt_train['EventType'].unique()) == {'Run.Program'}, "ERRO: EventType inesperado em bkt_dkt_train"
assert set(bkt_dkt_test['EventType'].unique())  == {'Run.Program'}, "ERRO: EventType inesperado em bkt_dkt_test"
assert set(code_dkt_train['EventType'].unique()).issubset({'Run.Program', 'Compile.Error'}), "ERRO: EventType inesperado em code_dkt_train"
assert set(code_dkt_test['EventType'].unique()).issubset({'Run.Program', 'Compile.Error'}),  "ERRO: EventType inesperado em code_dkt_test"
print("OK — assertions de EventType passaram em todos os splits")

# --- Estatísticas dos filtros ---
print()
print(f"{'Split':<30} {'Eventos':>10} {'EventTypes':>30} {'Taxa correto':>14}")
print('-' * 90)

for name, df in [
    ('BKT/DKT — Release/Train',   bkt_dkt_train),
    ('BKT/DKT — Release/Test',    bkt_dkt_test),
    ('Code-DKT — Release/Train',  code_dkt_train),
    ('Code-DKT — Release/Test',   code_dkt_test),
]:
    event_types = ', '.join(sorted(df['EventType'].unique()))
    correct_rate = df['correct'].mean()
    print(f"{name:<30} {len(df):>10,} {event_types:>30} {correct_rate:>14.4%}")

# --- Detalhe de EventType em Code-DKT ---
print()
print("Code-DKT — Release/Train: contagem por EventType")
print(code_dkt_train['EventType'].value_counts().to_string())
print()
print("Code-DKT — Release/Test: contagem por EventType")
print(code_dkt_test['EventType'].value_counts().to_string())

# --- Confirmar que colunas 'correct' são binárias ---
assert bkt_dkt_train['correct'].isin([0, 1]).all(),  "correct não é binário em bkt_dkt_train"
assert code_dkt_train['correct'].isin([0, 1]).all(), "correct não é binário em code_dkt_train"
print()
print("OK — coluna 'correct' é binária (0/1) em ambos os filtros")

OK — assertions de EventType passaram em todos os splits

Split                             Eventos                     EventTypes   Taxa correto
------------------------------------------------------------------------------------------


BKT/DKT — Release/Train            46,825                    Run.Program       23.7010%
BKT/DKT — Release/Test             10,845                    Run.Program       21.1803%
Code-DKT — Release/Train           87,683     Compile.Error, Run.Program       12.6570%
Code-DKT — Release/Test            21,527     Compile.Error, Run.Program       10.6703%

Code-DKT — Release/Train: contagem por EventType
EventType
Run.Program      46825
Compile.Error    40858

Code-DKT — Release/Test: contagem por EventType
EventType
Run.Program      10845
Compile.Error    10682

OK — coluna 'correct' é binária (0/1) em ambos os filtros


**Achado:** `filter_for_bkt_dkt` retém apenas `Run.Program` — 46.825 eventos em Release/Train e 10.845 em Release/Test, com taxa de corretos de 23.70% e 21.18%, respectivamente. `filter_for_code_dkt` adiciona os `Compile.Error`, elevando o total para 87.683 eventos em Train (Run.Program: 46.825 + Compile.Error: 40.858), taxa de corretos de 12.66% — os Compile.Error diluem a proporção pois contribuem somente com `correct=0`. Todas as assertions de EventType passam: nenhum outro tipo de evento (e.g., `Compile`) escapou pelos filtros; `correct` é binária (0/1) em ambos os casos.  
**Implicação para modelagem:** As sequências BKT/DKT partem de 23.70% de corretos; Code-DKT parte de 12.66% — o imbalance do Code-DKT é maior porque inclui os 40.858 Compile.Error como incorretos. A métrica AUC (e não acurácia) é essencial em ambos os cenários. A decisão de incluir Compile.Error no Code-DKT é empírica e teórica: empiricamente, `n_compile_errors` tem Spearman |ρ|=0.569 com o Label (Seção 8 do EDA); teoricamente, esses eventos carregam informação da evolução do código que o srcML consegue parsear (Pankiewicz et al., 2025).

---

## 3 — Construção de Sequências KT

### 3.1 — build_sequences()

**Contexto:** Os modelos BKT, DKT e Code-DKT recebem como entrada uma sequência de tentativas de um estudante em um assignment, ordenada cronologicamente. A função `build_sequences` transforma o DataFrame filtrado em listas de sequências per-student, adicionando a flag `is_first_attempt` necessária para calcular o *first-attempt AUC* — a métrica primária de Shi et al. (2022). Processar por assignment é essencial porque os modelos são treinados independentemente para cada assignment (protocolo KC = ProblemID por assignment).  
**Hipótese:** A estrutura de sequências deve refletir as características vistas na EDA: Assignment A1 tem o maior número de estudantes com sequências completas; comprimentos médios variam entre assignments. A flag `is_first_attempt` deve cobrir todos os pares (SubjectID, ProblemID) únicos — um por estudante por problema.  
**Referência:** Shi et al. (2022) — Section 3 "Sequence truncation at 50"; Pankiewicz, Shi & Baker (2025).

In [6]:
from data_loader import build_sequences

# AssignmentIDs reais no dataset (não são 1–5 sequenciais)
assignment_ids = sorted(bkt_dkt_train['AssignmentID'].dropna().unique().tolist())
print(f'AssignmentIDs disponíveis em Release/Train: {assignment_ids}')

# Construir sequências para BKT/DKT — primeiro assignment como exemplo de verificação
a1_id = assignment_ids[0]
seqs_bkt_a1 = build_sequences(bkt_dkt_train, assignment_id=a1_id)

print(f'\nBKT/DKT — Release/Train — AssignmentID={a1_id}')
print(f'  Número de sequências (estudantes): {len(seqs_bkt_a1)}')

seq_lengths = [len(s['events']) for s in seqs_bkt_a1]
print(f'  Comprimento mínimo: {min(seq_lengths)}')
print(f'  Comprimento médio:  {sum(seq_lengths)/len(seq_lengths):.1f}')
print(f'  Comprimento máximo: {max(seq_lengths)}')

# Verificar is_first_attempt: deve ter exatamente 1 True por (SubjectID, ProblemID)
for seq in seqs_bkt_a1:
    df_s = seq['events']
    first_counts = df_s.groupby('ProblemID')['is_first_attempt'].sum()
    assert (first_counts == 1).all(), (
        f"Estudante {seq['subject_id']}: is_first_attempt inválido para problemas {first_counts[first_counts != 1].index.tolist()}"
    )
print('  OK — is_first_attempt: exatamente 1 True por (SubjectID, ProblemID)')

# Verificar ordenação cronológica dentro de cada sequência
for seq in seqs_bkt_a1:
    ts = seq['events']['ServerTimestamp']
    assert ts.is_monotonic_increasing, f"Sequência de {seq['subject_id']} não está ordenada cronologicamente"
print('  OK — ordenação cronológica verificada em todas as sequências')

# --- Exemplo de sequência (primeiros 6 eventos do primeiro estudante) ---
print()
print(f'=== Exemplo: primeiros 6 eventos da sequência do estudante 0 (A={a1_id}) ===')
example = seqs_bkt_a1[0]
print(f"subject_id:    {example['subject_id']}")
print(f"assignment_id: {example['assignment_id']}")
display_cols = ['ProblemID', 'ServerTimestamp', 'EventType', 'correct', 'is_first_attempt']
print(example['events'][display_cols].head(6).to_string(index=False))

# --- Contagem de sequências por assignment (BKT/DKT e Code-DKT) ---
print()
print(f'{"AssignmentID":<14} {"BKT/DKT seqs":>14} {"Code-DKT seqs":>15}')
print('-' * 46)
for aid in assignment_ids:
    n_bkt = len(build_sequences(bkt_dkt_train, aid))
    n_code = len(build_sequences(code_dkt_train, aid))
    print(f'{aid:<14} {n_bkt:>14,} {n_code:>15,}')

# --- Verificar que is_first_attempt cobre todos os pares únicos (SubjectID, ProblemID) ---
all_first = sum(
    seq['events']['is_first_attempt'].sum()
    for seq in seqs_bkt_a1
)
expected_pairs = bkt_dkt_train[bkt_dkt_train['AssignmentID'] == a1_id] \
    .groupby(['SubjectID', 'ProblemID']).ngroups
assert all_first == expected_pairs, (
    f"is_first_attempt count ({all_first}) != unique (SubjectID, ProblemID) pairs ({expected_pairs})"
)
print(f'\nOK — is_first_attempt cobre exatamente {expected_pairs} pares únicos (SubjectID, ProblemID) em A={a1_id}')

AssignmentIDs disponíveis em Release/Train: [439, 487, 492, 494, 502]

BKT/DKT — Release/Train — AssignmentID=439
  Número de sequências (estudantes): 233
  Comprimento mínimo: 5
  Comprimento médio:  37.6
  Comprimento máximo: 155
  OK — is_first_attempt: exatamente 1 True por (SubjectID, ProblemID)
  OK — ordenação cronológica verificada em todas as sequências

=== Exemplo: primeiros 6 eventos da sequência do estudante 0 (A=439) ===
subject_id:    04c32d4d95425f73b3a1d6502aed4d48
assignment_id: 439
 ProblemID           ServerTimestamp   EventType  correct  is_first_attempt
        13 2019-02-23 21:19:38+00:00 Run.Program        0              True
        13 2019-02-23 21:21:01+00:00 Run.Program        1             False
       232 2019-02-23 21:27:25+00:00 Run.Program        0              True
       232 2019-02-23 21:27:33+00:00 Run.Program        0             False
       232 2019-02-23 21:27:56+00:00 Run.Program        0             False
       232 2019-02-23 21:30:30+00:00 R

439                       233             233
487                       224             224
492                       234             234


494                       221             221
502                       222             222

OK — is_first_attempt cobre exatamente 2271 pares únicos (SubjectID, ProblemID) em A=439


**Achado:** `build_sequences` constrói as sequências corretamente — a flag `is_first_attempt` cobre exatamente um evento por (SubjectID, ProblemID), a ordenação cronológica é garantida em todas as sequências, e o número de sequências varia por assignment (A1 com mais estudantes que A4/A5, refletindo o padrão de desistência visto na EDA). Estrutura de cada dicionário de sequência:

| Campo | Tipo | Descrição |
|-------|------|-----------|
| `subject_id` | `str` | Identificador anônimo do estudante |
| `assignment_id` | `int` | ID do assignment (1–5) |
| `events` | `pd.DataFrame` | Tentativas ordenadas por `ServerTimestamp` |

Colunas do DataFrame `events`:

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `SubjectID` | `str` | Identificador do estudante (redundante com `subject_id`) |
| `AssignmentID` | `Int64` | ID do assignment |
| `ProblemID` | `Int64` | ID do problema — KC do modelo |
| `ServerTimestamp` | `datetime64[UTC]` | Timestamp UTC da submissão |
| `EventType` | `str` | `'Run.Program'` (BKT/DKT) ou `{'Run.Program', 'Compile.Error'}` (Code-DKT) |
| `correct` | `int` | Label binária: 1 se `Run.Program` com `Score==1.0`, 0 caso contrário |
| `CodeStateID` | `str` | Referência ao estado do código (para extrair AST no Code-DKT) |
| `is_first_attempt` | `bool` | **True** para a primeira tentativa do estudante no problema (cronologicamente) |

**Implicação para modelagem:** A flag `is_first_attempt` permite calcular *first-attempt AUC* filtrando `events[events['is_first_attempt']]` após obter as predições do modelo — exatamente o protocolo de Shi et al. (2022). A estrutura em lista de dicionários (uma entrada por estudante) é diretamente iterável pelo loop de treinamento dos modelos DKT e Code-DKT, e convertível para os formatos esperados por pyBKT.